# Walkie Data 准备 Notebook


| # | 步骤 | 说明 |
|---|---|---|
| 1 | 数据集声明 | 填写 `DATASETS` 列表与输出根目录 |
| 2 | 检测路径 | 规范化配置，检查本地缓存是否已存在 |
| 3 | 下载数据 | 配置 HF 端点后按需下载 |
| 4 | 数据探索 | pandas `.head()` 查看字段与样本 |
| 5 | 粗略估计 | 字符采样估算各数据集 token 量与占比 |
| 6 | 去重与清洗 | 质量过滤 + SHA-1 去重，写入 `clean_texts.jsonl` |
| 7 | 训练 tokenizer | 从已清洗文本分层抽样训练 `byte_bpe` tokenizer |
| 8 | 统计 token 量 | 精确 tokenize 并展示各数据集占比 |
| 9 | 分离退火数据 | 按 stage / quality 分配 main / anneal |
| 10 | 编码成 bin | 写入 `main.bin` / `anneal.bin` + 元数据 |
| 11 | 验证 | 验证 bin 长度并打印训练命令 |

**运行顺序：逐节向下执行，每节开头的配置块可按需调整。**


## 1. 环境导入

定位仓库根目录，导入现有下载、文本迭代和 tokenizer 工具。

In [2]:
from __future__ import annotations

import hashlib
import importlib
import json
import random
import re
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
from tqdm.auto import tqdm

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.tokenizer import build_tokenizer, load_tokenizer
from data.encode import iter_texts

download_mod = importlib.import_module("data.download")
DOWNLOAD_FN = getattr(download_mod, "download", None) or getattr(download_mod, "c", None)
if DOWNLOAD_FN is None:
    raise RuntimeError("data/download.py 中没有可调用的 download(...) 或兼容函数 c(...)")

print(f"ROOT = {ROOT}")
print(f"Using downloader: data.download.{DOWNLOAD_FN.__name__}")

ROOT = /data/ldyData/LLM-Walk-Through
Using downloader: data.download.download


## 1. 数据集声明

在此填写 `DATASETS` 列表与输出根目录。每个数据集支持的字段：

| 字段 | 必填 | 说明 |
|---|---|---|
| `repo_id` | ✓ | HF 仓库 ID，格式 `owner/name` |
| `name` | | 友好名称，默认由 `repo_id` 生成 |
| `cache_dir` | | 本地缓存目录，不填则自动放在 `SOURCES_DIR/<name>/` |
| `subset_name` | | 子集名，如 `"finemath-3plus"`，用于自动探测 shard 路径 |
| `split` | | 读取的 split，默认 `"train"` |
| `text_field` | | 文本字段名，如 `"text"` / `"content"`，`None` 则自动猜测 |
| `num_shards` | | 仅下载前 N 个 shard（配合 `subset_name` 使用） |
| `max_samples` | | 最多读取多少条（调试用） |
| `stage` | | `"main"` / `"anneal"` / `"auto"`，控制退火集分配优先级 |
| `quality_hint` | | 质量先验 0–1，影响退火集选取，默认 `0.5` |


## 2. 检测路径

规范化数据集配置（填充默认值、转换 `cache_dir` 为 `Path`），然后列出各数据集的本地文件状态。


In [63]:
# ── 输出目录（统一放置所有中间产物）──────────────────────────────────────
OUTPUT_DIR       = ROOT / "data/cache/walkie_code"
SOURCES_DIR      = OUTPUT_DIR / "sources"
INTERMEDIATE_DIR = OUTPUT_DIR / "intermediate"

TOKENIZER_PATH       = OUTPUT_DIR / "tokenizer.json"
CLEAN_JSONL          = INTERMEDIATE_DIR / "clean_texts.jsonl"        # 去重清洗后
SAMPLES_JSONL        = INTERMEDIATE_DIR / "filtered_samples.jsonl"   # 含 n_tokens
SPLIT_MANIFEST_JSONL = OUTPUT_DIR / "split_manifest.jsonl"
MAIN_BIN             = OUTPUT_DIR / "main.bin"
ANNEAL_BIN           = OUTPUT_DIR / "anneal.bin"
META_PATH            = OUTPUT_DIR / "data_meta.json"

RANDOM_SEED = 42

# ── 数据集清单 ──────────────────────────────────────────────────────────
# category 可选值：text（网页文本）/ math / code
# 去重和统计均在同一 category 内进行。
DATASETS: list[dict[str, Any]] = [
    # {
    #     "name": "opc_fineweb_math",
    #     "repo_id": "OpenCoder-LLM/opc-fineweb-math-corpus",
    #     "cache_dir": f"{SOURCES_DIR}/opc-fineweb-math-corpus",
    #     "text_field": "text",
    #     "stage": "main",
    #     "category": "text",
    # },
    # {
    #     "name": "opc_fineweb_code",
    #     "repo_id": "OpenCoder-LLM/opc-fineweb-code-corpus",
    #     "cache_dir": f"{SOURCES_DIR}/opc-fineweb-code-corpus",
    #     "text_field": "text",
    #     "stage": "main",
    #     "category": "text",
    # },
    {
        "name": "opc_annealing",
        "repo_id": "OpenCoder-LLM/opc-annealing-corpus",
        "cache_dir": f"{SOURCES_DIR}/opc-annealing-corpus",
        "text_field": "text",
        "stage": "anneal",
        "category": "code",
    },
    # {
    #     "name": "starcoderdata",
    #     "repo_id": "bigcode/starcoderdata",
    #     "subset_name": "python",
    #     "cache_dir": f"{SOURCES_DIR}/starcoderdata",
    #     "text_field": "content",
    #     "stage": "main",
    #     "category": "code",
    # },
    {
        "name": "finemath_plus4",
        "repo_id": "HuggingFaceTB/finemath",
        "subset_name": "finemath-4plus",
        "cache_dir": f"{SOURCES_DIR}/finemath-plus4",
        "text_field": "text",
        "stage": "main",
        "category": "math",
    },
    # {
    #     "name": "finemath_infwebmath",
    #     "repo_id": "HuggingFaceTB/finemath",
    #     "subset_name": "infiwebmath-3plus",
    #     "cache_dir": f"{SOURCES_DIR}/finemath-infwebmath",
    #     "text_field": "text",
    #     "stage": "main",
    #     "category": "text",
    # },
    {
        "name": "the_stack_v2_python",
        "repo_id": "Cyrile/dataset-the-stack-v2-dedup-sub",
        "subset_name": "Python",
        "cache_dir": f"{SOURCES_DIR}/dataset-the-stack-v2-dedup-sub",
        "text_field": "content",
        "stage": "main",
        "category": "code",
    },
    {
        "name": "stack_edu",
        "repo_id": "affjljoo3581/stack-edu",
        "cache_dir": f"{SOURCES_DIR}/stack-edu",
        "text_field": "text",
        "stage": "anneal",
        "category": "code",
    },
    {
        "name": "starcoderdata_python_edu",
        "repo_id": "jon-tow/starcoderdata-python-edu",
        "cache_dir": f"{SOURCES_DIR}/starcoderdata-python-edu",
        "text_field": "content",
        "stage": "main",
        "category": "code",
    },
    {
        "name": "fineweb_edu_100bt",
        "repo_id": "HuggingFaceFW/fineweb-edu",
        "subset_name": "sample/100BT",
        "cache_dir": f"{SOURCES_DIR}/fineweb-edu-100BT",
        "text_field": "text",
        "stage": "main",
        "category": "text",
    },
]

print(f"OUTPUT_DIR = {OUTPUT_DIR}")
print(f"DATASETS   = {len(DATASETS)} 个数据集")


OUTPUT_DIR = /data/ldyData/LLM-Walk-Through/data/cache/walkie_code
DATASETS   = 6 个数据集


In [4]:
def safe_name(value: str) -> str:
    value = value.replace("/", "__")
    value = re.sub(r"[^0-9A-Za-z_.-]+", "_", value)
    return value.strip("._-") or "dataset"


def normalize_dataset_cfg(ds: dict[str, Any]) -> dict[str, Any]:
    if not ds.get("repo_id"):
        raise ValueError(f"数据集缺少 repo_id: {ds}")
    out = dict(ds)
    out.setdefault("name",            safe_name(str(out["repo_id"])))
    out.setdefault("subset_name",     None)
    out.setdefault("split",           "train")
    out.setdefault("text_field",      None)
    out.setdefault("num_shards",      None)
    out.setdefault("allow_patterns",  None)
    out.setdefault("ignore_patterns", None)
    out.setdefault("max_samples",     None)
    out.setdefault("max_chars",       None)
    out.setdefault("stage",           "auto")
    out.setdefault("quality_hint",    0.5)
    out.setdefault("category",        "code")   # text / math / code
    if out["stage"] not in {"main", "anneal", "auto"}:
        raise ValueError(f"stage 必须是 main/anneal/auto: {out['stage']}")
    if out["category"] not in {"text", "math", "code"}:
        raise ValueError(f"category 必须是 text/math/code: {out['category']}")
    out["cache_dir"] = Path(out["cache_dir"]) if "cache_dir" in out else SOURCES_DIR / safe_name(out["name"])
    return out


def iter_dataset_texts(
    ds: dict[str, Any], *, max_samples: int | None = None, max_chars: int | None = None
) -> Iterable[str]:
    yield from iter_texts(
        ds["cache_dir"],
        split=ds.get("split", "train"),
        text_field=ds.get("text_field"),
        max_samples=max_samples if max_samples is not None else ds.get("max_samples"),
        max_chars=max_chars  if max_chars  is not None else ds.get("max_chars"),
    )


# 规范化，创建输出目录，固定随机种子
DATASETS = [normalize_dataset_cfg(ds) for ds in DATASETS]
random.seed(RANDOM_SEED);  np.random.seed(RANDOM_SEED)
for p in (OUTPUT_DIR, SOURCES_DIR, INTERMEDIATE_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ── 路径状态表 ────────────────────────────────────────────────────────
rows = []
for ds in DATASETS:
    cache    = Path(ds["cache_dir"])
    parquets = list(cache.rglob("*.parquet")) if cache.exists() else []
    jsonls   = list(cache.rglob("*.jsonl"))   if cache.exists() else []
    rows.append({
        "name":          ds["name"],
        "category":      ds["category"],
        "stage":         ds["stage"],
        "exists":        cache.exists(),
        "parquet_files": len(parquets),
        "jsonl_files":   len(jsonls),
        "cache_dir":     str(ds["cache_dir"]),
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    for r in rows:
        status = "✓" if r["exists"] else "✗"
        print(f"{status} [{r['category']:5s}|{r['stage']:6s}] {r['name']:35s}  parquet={r['parquet_files']}  jsonl={r['jsonl_files']}")
        print(f"         {r['cache_dir']}")


,name,category,stage,exists,parquet_files,jsonl_files,cache_dir
0,opc_annealing,code,anneal,True,0,191,/data/ldyData/LLM-Walk-Through/data/cache/walk...
1,starcoderdata,code,main,True,59,0,/data/ldyData/LLM-Walk-Through/data/cache/walk...
2,finemath_plus4,math,main,True,64,0,/data/ldyData/LLM-Walk-Through/data/cache/walk...
3,the_stack_v2_python,code,main,True,84,0,/data/ldyData/LLM-Walk-Through/data/cache/walk...
4,stack_edu,code,anneal,True,29,0,/data/ldyData/LLM-Walk-Through/data/cache/walk...
5,starcoderdata_python_edu,code,main,True,125,0,/data/ldyData/LLM-Walk-Through/data/cache/walk...
6,fineweb_edu_100bt,text,main,True,55,0,/data/ldyData/LLM-Walk-Through/data/cache/walk...


## 3. 下载数据

配置 HF 端点与并发数后，按 `DATASETS` 清单逐一下载。已下载的数据集会自动跳过。


In [4]:
# ── 下载配置 ────────────────────────────────────────────────────────────
HF_ENDPOINT = "https://hf-mirror.com"  # 不需要镜像时改成 None
HF_TOKEN    = None
MAX_WORKERS = 8


def write_dataset_meta(ds: dict[str, Any]) -> None:
    meta = {k: (str(v) if isinstance(v, Path) else v) for k, v in ds.items() if k != "cache_dir"}
    path = Path(ds["cache_dir"]) / "data_meta.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")


def download_dataset(ds: dict[str, Any]) -> Path:
    kwargs = dict(
        repo_id=ds["repo_id"],
        local_dir=ds["cache_dir"],
        repo_type="dataset",
        subset_name=ds.get("subset_name"),
        num_shards=ds.get("num_shards"),
        hf_endpoint=HF_ENDPOINT,
        token=HF_TOKEN,
        max_workers=MAX_WORKERS,
    )
    if ds.get("allow_patterns") is not None:
        kwargs["allow_patterns"] = ds["allow_patterns"]
    if ds.get("ignore_patterns") is not None:
        kwargs["ignore_patterns"] = ds["ignore_patterns"]
    snapshot_dir = DOWNLOAD_FN(**kwargs)
    write_dataset_meta(ds)
    return Path(snapshot_dir)


for ds in DATASETS:
    print(f"\n=== {ds['name']} ({ds['repo_id']}) stage={ds['stage']} ===")
    snapshot = download_dataset(ds)
    print(f"  local: {snapshot}")



=== opc_fineweb_code (OpenCoder-LLM/opc-fineweb-code-corpus) stage=main ===
[download] 准备 HF dataset OpenCoder-LLM/opc-fineweb-code-corpus (subset=None) -> /data/ldyData/LLM-Walk-Through/data/cache/walkie_code/sources/opc-fineweb-code-corpus


Fetching 512 files:   0%|          | 0/512 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 4. 数据探索

读取每个数据集的第一个 parquet/jsonl 文件，展示字段列表与前几行文本预览，确认 `text_field` 是否正确。


In [6]:
EXPLORE_HEAD_ROWS = 3   # 每个数据集展示多少行 head

for ds in DATASETS:
    cache = Path(ds["cache_dir"])
    parquets = sorted(cache.rglob("*.parquet")) if cache.exists() else []
    jsonls   = sorted(cache.rglob("*.jsonl"))   if cache.exists() else []
    files    = (parquets + jsonls)[:1]           # 只读第一个文件
    if not files:
        print(f"[{ds['name']}] 未找到 parquet/jsonl 文件，路径: {cache}")
        continue

    f = files[0]
    try:
        import pandas as pd
        if f.suffix == ".parquet":
            df = pd.read_parquet(f)
        else:
            df = pd.read_json(f, lines=True)
        print(f"\n{'='*60}")
        print(f"[{ds['name']}]  {f.relative_to(cache)}  shape={df.shape}")
        print(f"columns: {list(df.columns)}")
        # 截断长文本避免刷屏
        text_col = ds.get("text_field") or next(
            (c for c in df.columns if c in ("text", "content", "code", "document")), df.columns[0]
        )
        preview = df[[text_col]].head(EXPLORE_HEAD_ROWS).copy()
        preview[text_col] = preview[text_col].astype(str).str[:200] + " …"
        display(preview)
    except Exception as e:
        print(f"[{ds['name']}] 读取失败: {e}")



[opc_annealing]  algorithmic_corpus/1.jsonl  shape=(100000, 3)
columns: ['type', 'lang', 'text']


,text
0,\nWrite a python function to find the kth smal...
1,\nWrite a python function to find the intersec...
2,\nWrite a function to find the missing number ...



[starcoderdata]  python/train-00000-of-00059.parquet  shape=(57608, 5)
columns: ['max_stars_repo_path', 'max_stars_repo_name', 'max_stars_count', 'id', 'content']


,content
0,<gh_stars>1-10\nclass Solution:\n def final...
1,<filename>lib/variables/latent_variables/__ini...
2,# -*- coding: utf-8 -*-\n# coding=utf-8\nimpo...



[finemath_plus4]  finemath-4plus/train-00000-of-00064.parquet  shape=(15268, 16)
columns: ['url', 'fetch_time', 'content_mime_type', 'warc_filename', 'warc_record_offset', 'warc_record_length', 'text', 'token_count', 'char_count', 'metadata', 'score', 'int_score', 'crawl', 'snapshot_type', 'language', 'language_score']


,text
0,# What is the term”exclamation mark” in mathem...
9,### Theory:\n\nLet us draw the graph of the eq...
16,# Equation solver with square root\n\nThis Equ...



[the_stack_v2_python]  Python/train-00000-of-00084.parquet  shape=(60330, 28)
columns: ['blob_id', 'directory_id', 'path', 'content_id', 'detected_licenses', 'license_type', 'repo_name', 'snapshot_id', 'revision_id', 'branch_name', 'visit_date', 'revision_date', 'committer_date', 'github_id', 'star_events_count', 'fork_events_count', 'gha_license_id', 'gha_event_created_at', 'gha_created_at', 'gha_language', 'src_encoding', 'language', 'is_vendor', 'is_generated', 'length_bytes', 'extension', 'filename', 'content']


,content
0,# Generated by Django 4.1.9 on 2023-06-29 16:1...
1,"""""""Syncronizes cell Zookeeper with LDAP data.\..."
2,# coding:utf-8\n\n# strategy warpper\n\n\nclas...


KeyboardInterrupt: 

## 5. 粗略估计 token 数量与占比

对每个数据集随机采样 `ESTIMATE_SAMPLE_ROWS` 行，以 `avg_chars / CHARS_PER_TOKEN` 估算每行 token 数，给出各数据集的规模感和 token 占比参考，帮助在正式扫描前发现明显异常。


In [5]:
# 基于“采样均值 × 总 rows”的全量外推估算

EXTRAPOLATE_SAMPLE_ROWS = 100_000
EXTRAPOLATE_CHARS_PER_TOKEN = 4.0


def count_rows_in_file(file_path: Path) -> int:
    suffix = file_path.suffix.lower()
    if suffix == ".parquet":
        import pyarrow.parquet as pq

        return pq.ParquetFile(file_path).metadata.num_rows
    if suffix == ".jsonl":
        with file_path.open("r", encoding="utf-8") as handle:
            return sum(1 for line in handle if line.strip())
    if suffix == ".json" and file_path.name != "dataset_infos.json":
        payload = json.loads(file_path.read_text(encoding="utf-8"))
        if isinstance(payload, list):
            return len(payload)
        if isinstance(payload, dict) and isinstance(payload.get("data"), list):
            return len(payload["data"])
        return 1
    return 0


def dataset_source_files(cache_dir: Path) -> tuple[list[Path], list[Path], list[Path]]:
    parquets = sorted(cache_dir.rglob("*.parquet")) if cache_dir.exists() else []
    jsonls = sorted(cache_dir.rglob("*.jsonl")) if cache_dir.exists() else []
    jsons = [
        path
        for path in sorted(cache_dir.rglob("*.json"))
        if path.name != "dataset_infos.json"
    ] if cache_dir.exists() else []
    return parquets, jsonls, jsons


def count_dataset_rows(ds: dict[str, Any]) -> tuple[int, int, int, int]:
    cache = Path(ds["cache_dir"])
    if not cache.exists():
        return 0, 0, 0, 0
    parquets, jsonls, jsons = dataset_source_files(cache)
    total_rows = 0
    for file_path in [*parquets, *jsonls, *jsons]:
        total_rows += count_rows_in_file(file_path)
    return total_rows, len(parquets), len(jsonls), len(jsons)


def rough_token_estimate_extrapolated() -> None:
    rows = []
    for ds in DATASETS:
        cache = Path(ds["cache_dir"])
        total_rows, parquet_files, jsonl_files, json_files = count_dataset_rows(ds)
        if not cache.exists():
            rows.append({
                "category": ds["category"],
                "dataset": ds["name"],
                "stage": ds["stage"],
                "total_rows": 0,
                "sampled_rows": 0,
                "avg_chars/row": 0,
                "avg_bytes/row": 0,
                "parquet_files": 0,
                "jsonl_files": 0,
                "json_files": 0,
                "est_total_bytes": 0,
                "est_total_tokens": 0,
            })
            continue

        char_total = 0
        byte_total = 0
        sample_count = 0
        sample_limit = min(EXTRAPOLATE_SAMPLE_ROWS, total_rows) if total_rows else EXTRAPOLATE_SAMPLE_ROWS
        for text in iter_dataset_texts(ds, max_samples=sample_limit):
            char_total += len(text)
            byte_total += len(text.encode("utf-8"))
            sample_count += 1

        avg_chars = char_total / max(1, sample_count)
        avg_bytes = byte_total / max(1, sample_count)
        est_total_bytes = round(avg_bytes * total_rows)
        est_total_tokens = round((avg_chars * total_rows) / EXTRAPOLATE_CHARS_PER_TOKEN)
        rows.append({
            "category": ds["category"],
            "dataset": ds["name"],
            "stage": ds["stage"],
            "total_rows": total_rows,
            "sampled_rows": sample_count,
            "avg_chars/row": round(avg_chars),
            "avg_bytes/row": round(avg_bytes),
            "parquet_files": parquet_files,
            "jsonl_files": jsonl_files,
            "json_files": json_files,
            "est_total_bytes": est_total_bytes,
            "est_total_tokens": est_total_tokens,
        })

    import pandas as pd

    df = pd.DataFrame(rows)
    total_est_tokens = df["est_total_tokens"].sum()
    total_est_bytes = df["est_total_bytes"].sum()
    df["sample_%"] = [
        f"{sampled / total * 100:.3f}%" if total else "0.000%"
        for sampled, total in zip(df["sampled_rows"], df["total_rows"])
    ]
    df["est_tok_%"] = [
        f"{value / max(1, total_est_tokens) * 100:.1f}%"
        for value in df["est_total_tokens"]
    ]
    df["est_size_%"] = [
        f"{value / max(1, total_est_bytes) * 100:.1f}%"
        for value in df["est_total_bytes"]
    ]
    df["est_size_gib"] = (df["est_total_bytes"] / (1024 ** 3)).round(3)

    print("── 各数据集全量外推估算 ─────────────────────────────────────")
    display(df[[
        "category", "dataset", "stage", "total_rows", "sampled_rows", "sample_%",
        "avg_chars/row", "est_size_gib", "est_total_tokens", "est_tok_%", "est_size_%",
    ]])

    print("\n── 按 category 汇总 ───────────────────────────────────────")
    cat_df = (
        df.groupby("category")[["total_rows", "est_total_tokens", "est_total_bytes"]]
        .sum()
        .reset_index()
        .sort_values("est_total_tokens", ascending=False)
    )
    cat_df["est_tok_%"] = [
        f"{value / max(1, total_est_tokens) * 100:.1f}%"
        for value in cat_df["est_total_tokens"]
    ]
    cat_df["est_size_%"] = [
        f"{value / max(1, total_est_bytes) * 100:.1f}%"
        for value in cat_df["est_total_bytes"]
    ]
    cat_df["est_size_gib"] = (cat_df["est_total_bytes"] / (1024 ** 3)).round(3)
    display(cat_df[[
        "category", "total_rows", "est_size_gib", "est_total_tokens", "est_tok_%", "est_size_%",
    ]].rename(columns={"est_total_tokens": "est_tokens"}))


In [56]:

rough_token_estimate_extrapolated()


── 各数据集全量外推估算 ─────────────────────────────────────


,category,dataset,stage,total_rows,sampled_rows,sample_%,avg_chars/row,est_size_gib,est_total_tokens,est_tok_%,est_size_%
0,code,opc_annealing,anneal,997044,100000,10.030%,616,0.572,153609427,1.0%,1.0%
1,math,finemath_plus4,main,978752,100000,10.217%,5433,5.023,1329294214,8.3%,8.4%
2,code,the_stack_v2_python,main,5058940,100000,1.977%,5522,26.181,6983286915,43.6%,43.6%
3,code,starcoderdata_python_edu,main,4160162,100000,2.404%,3715,14.449,3864196635,24.1%,24.1%
4,text,fineweb_edu_100bt,main,2957575,100000,3.381%,4990,13.801,3689393299,23.0%,23.0%



── 按 category 汇总 ───────────────────────────────────────


,category,total_rows,est_size_gib,est_tokens,est_tok_%,est_size_%
0,code,10216146,41.202,11001092977,68.7%,68.6%
2,text,2957575,13.801,3689393299,23.0%,23.0%
1,math,978752,5.023,1329294214,8.3%,8.4%


## 6. 去重与清洗

对每个数据集执行清洗


In [6]:
import os
import glob
import pyarrow.parquet as pq

def collect_blob_ids(parquet_dir):
    blob_ids = set()
    parquet_files = glob.glob(os.path.join(parquet_dir, "*.parquet"))
    for f in parquet_files:
        try:
            table = pq.read_table(f, columns=["blob_id"])
            blob_ids.update(table["blob_id"].to_pylist())
        except Exception as e:
            print(f"读取 {f} 时出错: {e}")
    return blob_ids

dir1 = f"{SOURCES_DIR}/stack-edu/Python"
dir2 = f"{SOURCES_DIR}/dataset-the-stack-v2-dedup-sub/Python"

print("收集第一个目录的 blob_id ...")
ids1 = collect_blob_ids(dir1)
print(f"第一个目录共 {len(ids1)} 个 blob_id")

print("收集第二个目录的 blob_id ...")
ids2 = collect_blob_ids(dir2)
print(f"第二个目录共 {len(ids2)} 个 blob_id")

dupes = ids1 & ids2
print(f"\n重复的 blob_id 数量: {len(dupes)}")
if dupes:
    print("示例重复 blob_id:", list(dupes)[:10])

收集第一个目录的 blob_id ...
第一个目录共 25286012 个 blob_id
收集第二个目录的 blob_id ...
第二个目录共 18065153 个 blob_id

重复的 blob_id 数量: 9662005
示例重复 blob_id: ['346e8c0234036b681556914b2da74170c5550eb9', '2c2a4795e2495f428f534fcca696d99c89358a22', 'edb86e8ffe233edcc71b30809d084b9fa522e870', '24dbb55f2a95db7f860b01e9ec1b8a91d7e4436b', '4438cb0d1d5836260b3e5185bd6723ea3977c1d0', '4e1c05492516408a4dcbc073e03ebadd9d313e9c', '1419e07985f88c3a75b9e524b1f59632e787a7e6', '836a57dd34ccf693b06741d907b938e599058e81', 'af1da78173616ac5f69cdccb00a3f1ed2ce8097d', '280bb77a24ae09804749d769d946b6d9b9e82e2e']


In [7]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pyarrow as pa
import pyarrow.compute as pc

# ── 配置 ──────────────────────────────────────────────────────────
MAX_WORKERS = min(16, (os.cpu_count() or 4) * 2)
COMPRESSION = "zstd"

def _dedup_file(path: str, dupes_arr: pa.Array) -> tuple[str, int, int]:
    table = pq.read_table(path)
    before = len(table)
    # 过滤掉在 dupes 中的 blob_id
    mask = pc.invert(pc.is_in(table["blob_id"], value_set=dupes_arr))
    filtered = table.filter(mask)
    after = len(filtered)

    if after < before:
        tmp = path + ".tmp"
        pq.write_table(filtered, tmp, compression=COMPRESSION)
        os.replace(tmp, path)
    return path, before, after

def remove_duplicates(parquet_dir: str, dupes_set: set) -> None:
    if not dupes_set:
        print("无重复 blob_id，跳过。")
        return

    files = glob.glob(os.path.join(parquet_dir, "*.parquet"))
    # 构造 Arrow Array (假设类型为 string)
    dupes_arr = pa.array(list(dupes_set), type=pa.string())
    
    total_removed = 0
    print(f"正在从 {Path(parquet_dir).name} 中并行删除重复项...")
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(_dedup_file, f, dupes_arr) for f in files]
        for fut in as_completed(futures):
            path, before, after = fut.result()
            if before > after:
                total_removed += (before - after)
                print(f"  {Path(path).name}: {before} -> {after} (删除了 {before-after} 条)")

    print(f"\n去重完成。共从 dir1 删除 {total_removed} 条记录。")

if "dupes" in globals() and dupes:
    remove_duplicates(dir1, dupes)
else:
    print("错误: 未在内存中找到 'dupes' 集合，请确保运行了上一个单元格。")

正在从 Python 中并行删除重复项...
  00018.parquet: 871931 -> 539435 (删除了 332496 条)
  00002.parquet: 871932 -> 539332 (删除了 332600 条)
  00014.parquet: 871931 -> 537505 (删除了 334426 条)
  00013.parquet: 871931 -> 538706 (删除了 333225 条)
  00006.parquet: 871932 -> 538771 (删除了 333161 条)
  00026.parquet: 871931 -> 537932 (删除了 333999 条)
  00008.parquet: 871932 -> 539063 (删除了 332869 条)
  00019.parquet: 871931 -> 538399 (删除了 333532 条)
  00007.parquet: 871932 -> 539092 (删除了 332840 条)
  00004.parquet: 871932 -> 539495 (删除了 332437 条)
  00020.parquet: 871931 -> 539014 (删除了 332917 条)
  00027.parquet: 871931 -> 538510 (删除了 333421 条)
  00010.parquet: 871932 -> 538040 (删除了 333892 条)
  00017.parquet: 871931 -> 538776 (删除了 333155 条)
  00012.parquet: 871932 -> 538496 (删除了 333436 条)
  00025.parquet: 871931 -> 538355 (删除了 333576 条)
  00011.parquet: 871932 -> 539559 (删除了 332373 条)
  00023.parquet: 871931 -> 538378 (删除了 333553 条)
  00021.parquet: 871931 -> 538231 (删除了 333700 条)
  00000.parquet: 871932 -> 538680 (删除了 333252 

In [ ]:
from pathlib import Path

import pyarrow.compute as pc
import pyarrow.parquet as pq

path = Path(f"{SOURCES_DIR}/starcoderdata")
SKIP_DIR_NAMES = {"python"}
PARQUET_FILES = sorted(path.rglob("*.parquet"))

if not PARQUET_FILES:
    raise FileNotFoundError(f"没有找到 parquet 文件: {path}")

total_before = 0
total_after = 0
total_removed = 0

for parquet_path in PARQUET_FILES:
    table = pq.read_table(parquet_path)

    if "int_score" not in table.column_names:
        raise KeyError(f"{parquet_path} 缺少 int_score 列")

    before = len(table)
    mask = pc.greater_equal(table["int_score"], 4)
    filtered = table.filter(mask)
    after = len(filtered)
    removed = before - after

    tmp_path = parquet_path.with_suffix(".parquet.tmp")
    pq.write_table(filtered, tmp_path, compression="zstd")
    tmp_path.replace(parquet_path)

    total_before += before
    total_after += after
    total_removed += removed

    print(f"{parquet_path}: {before} -> {after}，删除 {removed} 条")

print("\n处理完成")
print(f"总样本数: {total_before} -> {total_after}")
print(f"总删除数: {total_removed}")

/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/sources/temp/C/00000.parquet: 974728 -> 230579，删除 744149 条
/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/sources/temp/C/00001.parquet: 974728 -> 231099，删除 743629 条
/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/sources/temp/C/00002.parquet: 974728 -> 230099，删除 744629 条
/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/sources/temp/C/00003.parquet: 974728 -> 230349，删除 744379 条
/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/sources/temp/C/00004.parquet: 974727 -> 231341，删除 743386 条
/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/sources/temp/C/00005.parquet: 974727 -> 231117，删除 743610 条
/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/sources/temp/CSharp/00000.parquet: 761667 -> 65732，删除 695935 条
/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/sources/temp/CSharp/00001.parquet: 761667 -> 65790，删除 695877 条
/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/sources/temp/CSharp/00002.parquet:

In [ ]:
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import pyarrow.compute as pc
import pyarrow.parquet as pq


# ── 配置 ────────────────────────────────────────────────────────────────────
STARCODER_ROOT  = Path(f"{SOURCES_DIR}/starcoderdata")
SKIP_DIR_NAMES  = {"python"}    # 已由仓库级去重单独处理
COMPRESSION     = "zstd"
MAX_WORKERS     = min(16, os.cpu_count() or 4)   # 进程数，按服务器 CPU 数自动调整


# ── 单文件处理函数（跑在子进程里，不能引用 notebook 全局变量）──────────────────
def _filter_one_file(args: tuple[str, str]) -> tuple[str, int, int]:
    """
    过滤单个 parquet 文件：删除 max_stars_count == 0 的行。
    返回 (file_path, before, after)。
    """
    parquet_path_str, compression = args
    import pyarrow.compute as _pc
    import pyarrow.parquet as _pq
    from pathlib import Path

    parquet_path = Path(parquet_path_str)
    try:
        table = _pq.read_table(parquet_path)
    except Exception as exc:
        print(f"  [ERR] 读取失败 {parquet_path.name}: {exc}", flush=True)
        return parquet_path_str, 0, 0

    if "max_stars_count" not in table.column_names:
        return parquet_path_str, len(table), len(table)   # 无该列，原样保留

    before = len(table)
    keep_mask = _pc.or_(
        _pc.is_null(table["max_stars_count"]),
        _pc.not_equal(table["max_stars_count"], 0),
    )
    filtered = table.filter(keep_mask)
    after    = len(filtered)

    if after < before:                          # 有删除才写
        tmp = parquet_path.with_suffix(".parquet.tmp")
        _pq.write_table(filtered, tmp, compression=compression)
        tmp.replace(parquet_path)

    return parquet_path_str, before, after


# ── 收集目标文件 ─────────────────────────────────────────────────────────────
target_subdirs = [
    d for d in STARCODER_ROOT.iterdir()
    if d.is_dir() and not d.name.startswith(".") and d.name.lower() not in SKIP_DIR_NAMES
] if STARCODER_ROOT.exists() else []

if not target_subdirs:
    raise RuntimeError(f"没有可处理的子目录: {STARCODER_ROOT}")

all_files: list[Path] = []
for subdir in sorted(target_subdirs):
    files = sorted(subdir.rglob("*.parquet"))
    print(f"  [{subdir.name}] {len(files)} 个 parquet 文件")
    all_files.extend(files)

print(f"\n共 {len(all_files)} 个文件，使用 {MAX_WORKERS} 进程并行处理 …\n")

# ── 并行执行 ─────────────────────────────────────────────────────────────────
total_before = total_after = 0
tasks = [(str(f), COMPRESSION) for f in all_files]

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(_filter_one_file, t): t[0] for t in tasks}
    for future in tqdm(as_completed(futures), total=len(futures),
                       desc="filter max_stars_count=0", unit="file"):
        path_str, before, after = future.result()
        total_before += before
        total_after  += after
        removed = before - after
        if removed:
            print(f"  {Path(path_str).name}: {before} -> {after}，删除 {removed} 条")

total_removed = total_before - total_after
print(f"\n处理完成")
print(f"处理文件数  : {len(all_files)}")
print(f"总样本数    : {total_before:,} -> {total_after:,}")
print(f"总删除数    : {total_removed:,}  ({total_removed / max(1, total_before) * 100:.2f}%)")



== 处理子目录: /data/ldyData/LLM-Walk-Through/data/cache/walkie_code/sources/starcoderdata/c  文件数=53 ==
  train-00000-of-00053.parquet: 161072 -> 100399，删除 60673 条
  train-00001-of-00053.parquet: 161072 -> 100106，删除 60966 条
  train-00002-of-00053.parquet: 161072 -> 100140，删除 60932 条
  train-00003-of-00053.parquet: 161072 -> 100584，删除 60488 条
  train-00004-of-00053.parquet: 161072 -> 100192，删除 60880 条
  train-00005-of-00053.parquet: 161072 -> 100117，删除 60955 条
  train-00006-of-00053.parquet: 161072 -> 100691，删除 60381 条
  train-00007-of-00053.parquet: 161072 -> 100027，删除 61045 条
  train-00008-of-00053.parquet: 161072 -> 100397，删除 60675 条
  train-00009-of-00053.parquet: 161072 -> 99962，删除 61110 条
  train-00010-of-00053.parquet: 161072 -> 100096，删除 60976 条
  train-00011-of-00053.parquet: 161072 -> 99938，删除 61134 条
  train-00012-of-00053.parquet: 161072 -> 99981，删除 61091 条
  train-00013-of-00053.parquet: 161072 -> 100730，删除 60342 条
  train-00014-of-00053.parquet: 161072 -> 99986，删除 61086 条
  tr

In [9]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

TARGET_DIR = Path(f"{SOURCES_DIR}/starcoderdata-python-edu/data")
REF_DIRS = [
    Path(f"{SOURCES_DIR}/stack-edu/Python"),
    Path(f"{SOURCES_DIR}/dataset-the-stack-v2-dedup-sub/Python"),
]
REF_REPO_COL = "repo_name"
TARGET_REPO_COL = "max_stars_repo_name"
REPO_DEDUP_COMPRESSION = "zstd"


def collect_repo_names(dirs: list[Path], col: str = REF_REPO_COL) -> frozenset[str]:
    """从多个目录下收集 repo 名，用于目标数据集去重。"""
    repos: set[str] = set()
    for directory in dirs:
        files = sorted(directory.rglob("*.parquet")) if directory.exists() else []
        for file_path in files:
            try:
                table = pq.read_table(file_path, columns=[col])
                values = table[col].to_pylist()
                repos.update(value for value in values if value is not None)
            except Exception as exc:
                print(f"  [warn] 读取 {file_path} 失败: {exc}")
        print(f"  [{directory.name}] 累计 {col} 数 = {len(repos):,}")
    return frozenset(repos)


def dedup_parquets_by_repo(
    target_dir: Path,
    ref_repos: frozenset[str],
    src_col: str = TARGET_REPO_COL,
) -> dict[str, int]:
    """原地删除 target_dir 中 repo 与参考集合重复的记录。"""
    files = sorted(target_dir.rglob("*.parquet")) if target_dir.exists() else []
    if not files:
        raise FileNotFoundError(f"未找到 parquet 文件: {target_dir}")

    ref_set = set(ref_repos)
    total_before = total_after = total_removed = skipped_files = 0
    for file_path in tqdm(files, desc=f"repo dedup {target_dir.name}", unit="file"):
        try:
            df = pd.read_parquet(file_path)
        except Exception as exc:
            skipped_files += 1
            print(f"  [skip] 读取失败 {file_path.name}: {exc}")
            continue

        if src_col not in df.columns:
            skipped_files += 1
            print(f"  [skip] {file_path.name} 缺少列 {src_col}")
            continue

        before = len(df)
        keep_mask = df[src_col].isna() | ~df[src_col].isin(ref_set)
        filtered = df[keep_mask]
        after = len(filtered)
        removed = before - after

        if removed > 0:
            tmp_path = file_path.with_suffix(".parquet.tmp")
            filtered.to_parquet(tmp_path, compression=REPO_DEDUP_COMPRESSION, index=False)
            tmp_path.replace(file_path)

        total_before += before
        total_after += after
        total_removed += removed

    return {
        "files": len(files),
        "rows_before": total_before,
        "rows_after": total_after,
        "rows_removed": total_removed,
        "ref_repos": len(ref_repos),
        "skipped_files": skipped_files,
    }


print("Step 1: 收集参考数据集的 repo_name …")
ref_repos = collect_repo_names(REF_DIRS, col=REF_REPO_COL)
print(f"参考 repo_name 总数 = {len(ref_repos):,}\n")

print(f"Step 2: 过滤 {TARGET_DIR} …")
dedup_result = dedup_parquets_by_repo(TARGET_DIR, ref_repos, src_col=TARGET_REPO_COL)

print("\n── 结果 ────────────────────────────────────────────────────────────────")
for key, value in dedup_result.items():
    print(f"  {key:20s} = {value:,}" if isinstance(value, int) else f"  {key:20s} = {value}")

Step 1: 收集参考数据集的 repo_name …
  [Python] 累计 repo_name 数 = 942,791
  [Python] 累计 repo_name 数 = 1,840,576
参考 repo_name 总数 = 1,840,576

Step 2: 过滤 /data/ldyData/LLM-Walk-Through/data/cache/walkie_code/sources/starcoderdata-python-edu/data …


repo dedup data:   0%|          | 0/125 [00:00<?, ?file/s]


── 结果 ────────────────────────────────────────────────────────────────
  files                = 125
  rows_before          = 12,866,649
  rows_after           = 10,452,810
  rows_removed         = 2,413,839
  ref_repos            = 1,840,576
  skipped_files        = 0


In [34]:
from pathlib import Path

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm.auto import tqdm

FINEWEB_EDU_DIR = Path(f"{SOURCES_DIR}/fineweb-edu-100BT/sample/100BT")
SCORE_COL_CANDIDATES = ("scores", "score")
SCORE_THRESHOLD = 3.7
FINEWEB_COMPRESSION = "zstd"


def resolve_score_column(column_names: list[str]) -> str:
    for column_name in SCORE_COL_CANDIDATES:
        if column_name in column_names:
            return column_name
    raise KeyError(f"未找到分数字段，候选列: {SCORE_COL_CANDIDATES}，实际列: {column_names}")


parquet_files = sorted(FINEWEB_EDU_DIR.rglob("*.parquet")) if FINEWEB_EDU_DIR.exists() else []
if not parquet_files:
    raise FileNotFoundError(f"没有找到 parquet 文件: {FINEWEB_EDU_DIR}")

total_before = 0
total_after = 0
total_removed = 0
score_column = None

for parquet_path in tqdm(parquet_files, desc="filter fineweb-edu score>=3.5", unit="file"):
    table = pq.read_table(parquet_path)
    if score_column is None:
        score_column = resolve_score_column(table.column_names)
        print(f"score column = {score_column}")
    elif score_column not in table.column_names:
        raise KeyError(f"{parquet_path} 缺少列 {score_column}")

    before = len(table)
    score_array = pc.cast(table[score_column], pa.float64(), safe=False)
    keep_mask = pc.greater_equal(score_array, SCORE_THRESHOLD)
    filtered = table.filter(keep_mask)
    after = len(filtered)
    removed = before - after

    if removed > 0:
        tmp_path = parquet_path.with_suffix(".parquet.tmp")
        pq.write_table(filtered, tmp_path, compression=FINEWEB_COMPRESSION)
        tmp_path.replace(parquet_path)

    total_before += before
    total_after += after
    total_removed += removed

print("\n处理完成")
print(f"文件数: {len(parquet_files)}")
print(f"总样本数: {total_before:,} -> {total_after:,}")
print(f"总删除数: {total_removed:,}")
print(f"保留阈值: {score_column} >= {SCORE_THRESHOLD}")

filter fineweb-edu score>=3.5:   0%|          | 0/57 [00:00<?, ?file/s]

score column = score

处理完成
文件数: 57
总样本数: 4,026,307 -> 2,957,575
总删除数: 1,068,732
保留阈值: score >= 3.7


In [55]:
from pathlib import Path

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm.auto import tqdm

FINEWEB_EDU_DIR = Path(f"{SOURCES_DIR}/starcoderdata-python-edu")
SCORE_COL_CANDIDATES = ("score")
SCORE_THRESHOLD = 2.5
FINEWEB_COMPRESSION = "zstd"


def resolve_score_column(column_names: list[str]) -> str:
    for column_name in SCORE_COL_CANDIDATES:
        if column_name in column_names:
            return column_name
    raise KeyError(f"未找到分数字段，候选列: {SCORE_COL_CANDIDATES}，实际列: {column_names}")


parquet_files = sorted(FINEWEB_EDU_DIR.rglob("*.parquet")) if FINEWEB_EDU_DIR.exists() else []
if not parquet_files:
    raise FileNotFoundError(f"没有找到 parquet 文件: {FINEWEB_EDU_DIR}")

total_before = 0
total_after = 0
total_removed = 0
score_column = "score"

for parquet_path in tqdm(parquet_files, desc="filter starcoderdata-python-edu score>=2", unit="file"):
    table = pq.read_table(parquet_path)
    if score_column is None:
        score_column = resolve_score_column(table.column_names)
        print(f"score column = {score_column}")
    elif score_column not in table.column_names:
        raise KeyError(f"{parquet_path} 缺少列 {score_column}")

    before = len(table)
    score_array = pc.cast(table[score_column], pa.float64(), safe=False)
    keep_mask = pc.greater_equal(score_array, SCORE_THRESHOLD)
    filtered = table.filter(keep_mask)
    after = len(filtered)
    removed = before - after

    if removed > 0:
        tmp_path = parquet_path.with_suffix(".parquet.tmp")
        pq.write_table(filtered, tmp_path, compression=FINEWEB_COMPRESSION)
        tmp_path.replace(parquet_path)

    total_before += before
    total_after += after
    total_removed += removed

print("\n处理完成")
print(f"文件数: {len(parquet_files)}")
print(f"总样本数: {total_before:,} -> {total_after:,}")
print(f"总删除数: {total_removed:,}")
print(f"保留阈值: {score_column} >= {SCORE_THRESHOLD}")

filter starcoderdata-python-edu score>=2:   0%|          | 0/125 [00:00<?, ?file/s]


处理完成
文件数: 125
总样本数: 4,160,162 -> 4,160,162
总删除数: 0
保留阈值: score >= 2.5


### 二次清洗(缩减到22B tokens) 

In [ ]:
import json
from pathlib import Path

root = Path('.')
files = sorted(root.rglob('*.jsonl'))

before = 0
after = 0
malformed = 0

for fp in files:
    tmp = fp.with_suffix(fp.suffix + '.tmp')
    kept = 0
    total = 0
    with fp.open('r', encoding='utf-8') as fin, tmp.open('w', encoding='utf-8') as fout:
        for line in fin:
            if not line.strip():
                continue
            total += 1
            try:
                obj = json.loads(line)
            except Exception:
                malformed += 1
                continue
            lang = str(obj.get('lang', '')).strip().lower()
            if lang == 'python':
                fout.write(json.dumps(obj, ensure_ascii=False) + '\n')
                kept += 1
    tmp.replace(fp)
    before += total
    after += kept

print(f'Files processed: {len(files)}')
print(f'Total records before: {before}')
print(f'Total records after (lang=python): {after}')
print(f'Removed records: {before - after}')
print(f'Malformed skipped: {malformed}')

## 7. 分层抽样训练 tokenizer



In [17]:
# ── tokenizer 训练配置 ───────────────────────────────────────────────────
TOKENIZER_KIND                    = "byte_bpe"
VOCAB_SIZE                        = 65_536
SPECIAL_TOKENS                    = ["<|endoftext|>", "<|pad|>"]
EOS_TOKEN                         = "<|endoftext|>"
PAD_TOKEN                         = "<|pad|>"
TOKENIZER_TRAIN_MAX_CHARS         = 600_000_000     # 训练语料总字符上限（直接读原始文件）
TOKENIZER_TRAIN_MAX_CHARS_PER_DS  = 150_000_000      # 单数据集字符上限（分层限制）
TOKENIZER_TRAIN_BATCH_SIZE        = 1_000           # 每批送给 HF trainer 的文本数
FORCE_RETRAIN_TOKENIZER           = False           # True 强制重训，忽略已有 tokenizer

# ── 分层抽样：直接从各数据集原始文件读取 ────────────────────────────────────────
# 不依赖 clean_texts.jsonl；大文件场景下内存安全。

import itertools

def collect_tokenizer_corpus_batched() -> Iterable[list[str]]:
    """
    直接从各数据集 cache_dir 分层随机抽样，按 TOKENIZER_TRAIN_BATCH_SIZE 批量 yield。
    不把全量语料加载进内存。
    """
    rng = random.Random(RANDOM_SEED)
    total_chars = 0
    batch: list[str] = []

    for ds in DATASETS:
        if total_chars >= TOKENIZER_TRAIN_MAX_CHARS:
            break
        ds_chars = 0
        ds_samples = 0
        texts_buf = list(itertools.islice(iter_dataset_texts(ds), 50_000))
        rng.shuffle(texts_buf)

        for raw_text in texts_buf:
            remaining_total = TOKENIZER_TRAIN_MAX_CHARS - total_chars
            remaining_ds    = TOKENIZER_TRAIN_MAX_CHARS_PER_DS - ds_chars
            take = min(len(raw_text), remaining_total, remaining_ds)
            if take <= 0:
                break
            batch.append(raw_text[:take])
            total_chars += take
            ds_chars    += take
            ds_samples  += 1
            if len(batch) >= TOKENIZER_TRAIN_BATCH_SIZE:
                yield batch
                batch = []
            if total_chars >= TOKENIZER_TRAIN_MAX_CHARS or ds_chars >= TOKENIZER_TRAIN_MAX_CHARS_PER_DS:
                break

        print(f"  [{ds['category']}] {ds['name']}: chars={ds_chars:,}  samples={ds_samples:,}")
        if total_chars >= TOKENIZER_TRAIN_MAX_CHARS:
            break

    if batch:
        yield batch
    print(f"总 tokenizer 训练字符 ≈ {total_chars:,}")


# ── 训练或加载 tokenizer ─────────────────────────────────────────────────
# 使用 HuggingFace `tokenizers` 库的 ByteLevelBPETokenizer（Rust 实现，速度远快于纯 Python）。
# 保存格式：
#   <TOKENIZER_PATH>/vocab.json + merges.txt  （ByteLevelBPETokenizer 原生格式）
#   <TOKENIZER_PATH>/tokenizer.json            （完整序列化，HFTokenizer.from_file 可直接加载）

from tokenizers import Tokenizer as HFTokenizer

if TOKENIZER_PATH.exists() and (TOKENIZER_PATH / "tokenizer.json").exists() and not FORCE_RETRAIN_TOKENIZER:
    hf_tok = HFTokenizer.from_file(str(TOKENIZER_PATH / "tokenizer.json"))
    print(f"[tokenizer] loaded: {TOKENIZER_PATH}  vocab={hf_tok.get_vocab_size()}")

else:
    from tokenizers import ByteLevelBPETokenizer

    hf_bpe = ByteLevelBPETokenizer()
    print("开始训练 ByteLevelBPETokenizer")
    hf_bpe.train_from_iterator(
        collect_tokenizer_corpus_batched(),
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        show_progress=True,
    )
    TOKENIZER_PATH.mkdir(parents=True, exist_ok=True)
    hf_bpe.save_model(str(TOKENIZER_PATH))                    # vocab.json + merges.txt
    hf_tok = hf_bpe._tokenizer
    hf_tok.save(str(TOKENIZER_PATH / "tokenizer.json"))       # 完整序列化
    print(f"[tokenizer] saved: {TOKENIZER_PATH}  vocab={hf_tok.get_vocab_size()}")

# ── special token / bin dtype 校验 ───────────────────────────────────────
EOS_ID = hf_tok.token_to_id(EOS_TOKEN)
PAD_ID = hf_tok.token_to_id(PAD_TOKEN)
if EOS_ID is None or PAD_ID is None:
    raise RuntimeError(f"tokenizer 必须只包含并识别 {SPECIAL_TOKENS}，得到 eos={EOS_ID}, pad={PAD_ID}")

_vocab_size = hf_tok.get_vocab_size()
DTYPE = np.dtype(np.uint16 if _vocab_size <= np.iinfo(np.uint16).max + 1 else np.uint32)
if _vocab_size - 1 > np.iinfo(DTYPE).max:
    raise RuntimeError(f"vocab_size={_vocab_size} 超出 dtype={DTYPE} 可表示范围")
print(f"vocab_size={_vocab_size}  eos_id={EOS_ID}  pad_id={PAD_ID}  bin dtype={DTYPE}")

# ── 简单验证 ──────────────────────────────────────────────────────────────────
_sample = "def hello(world: str) -> None:\n    print(f'hello {world}')\n"
_enc    = hf_tok.encode(_sample)
print(f"[验证] token 数={len(_enc.ids)}  前 20 ids={_enc.ids[:20]}")


开始训练 ByteLevelBPETokenizer
  [code] opc_annealing: chars=30,746,861  samples=50,000
  [math] finemath_plus4: chars=150,000,000  samples=27,395
  [code] the_stack_v2_python: chars=150,000,000  samples=26,611
  [code] starcoderdata_python_edu: chars=150,000,000  samples=31,115
  [text] fineweb_edu_100bt: chars=119,253,139  samples=23,297
总 tokenizer 训练字符 ≈ 600,000,000



[tokenizer] saved: /data/ldyData/LLM-Walk-Through/data/cache/walkie_code/tokenizer.json  vocab=65536
vocab_size=65536  eos_id=0  pad_id=1  bin dtype=uint16
[验证] token 数=20  前 20 ids=[553, 21907, 9, 4573, 27, 615, 10, 1372, 678, 27, 296, 769, 9, 71, 8, 4810, 509, 4573, 7484, 200]


## 9. 分离退火数据

按显式评分规则分配到 `main` / `anneal` 两个阶段：
- `opc_annealing` 全量进入 `anneal`；
- `starcoderdata_python_edu` 中 `int_score >= 4` 进入 `anneal`，其余进入 `main`；
- `the_stack_v2_python` 中 `star_events_count >= 30` 进入 `anneal`，其余进入 `main`；
- `stack_edu` 中 `int_score == 4` 进入 `anneal`，其余进入 `main`；
- 其他数据集全量进入 `main`。

本节会同时枚举后续编码任务，并给出整理后的阶段 token 粗略估计。

In [58]:
# ── Step 9：按显式评分规则分离 main / anneal ──────────────────────────────
# anneal:
#   - opc_annealing 全量
#   - starcoderdata_python_edu 中 int_score >= 4
#   - the_stack_v2_python 中 star_events_count >= 30
#   - stack_edu 中 int_score == 4
# main:
#   - 其余数据集全量
#   - starcoderdata_python_edu 中 int_score < 4 或缺失
#   - the_stack_v2_python 中 star_events_count < 30 或缺失
#   - stack_edu 中 int_score != 4 或缺失

import json
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm.auto import tqdm

SHARDS_DIR = INTERMEDIATE_DIR / "shards"
STAGE_ESTIMATE_SAMPLE_ROWS = 20_000
STAGE_ESTIMATE_CHARS_PER_TOKEN = 4.0

SPLIT_RULES_BY_DATASET: dict[str, list[dict[str, Any]]] = {
    "opc_annealing": [
        {
            "stage": "anneal",
            "mode": "all",
            "description": "full dataset",
        },
    ],
    "starcoderdata_python_edu": [
        {
            "stage": "anneal",
            "mode": "parquet_filter",
            "column": "int_score",
            "op": "ge",
            "value": 4,
            "include_null": False,
            "description": "int_score >= 4",
        },
        {
            "stage": "main",
            "mode": "parquet_filter",
            "column": "int_score",
            "op": "lt",
            "value": 4,
            "include_null": True,
            "description": "int_score < 4 or null",
        },
    ],
    "the_stack_v2_python": [
        {
            "stage": "anneal",
            "mode": "parquet_filter",
            "column": "star_events_count",
            "op": "ge",
            "value": 30,
            "include_null": False,
            "description": "star_events_count >= 30",
        },
        {
            "stage": "main",
            "mode": "parquet_filter",
            "column": "star_events_count",
            "op": "lt",
            "value": 30,
            "include_null": True,
            "description": "star_events_count <30 or null",
        },
    ],
    "stack_edu": [
        {
            "stage": "anneal",
            "mode": "parquet_filter",
            "column": "int_score",
            "op": "eq",
            "value": 4,
            "include_null": False,
            "description": "int_score == 4",
        },
        {
            "stage": "main",
            "mode": "parquet_filter",
            "column": "int_score",
            "op": "ne",
            "value": 4,
            "include_null": True,
            "description": "int_score != 4 or null",
        },
    ],
}
DEFAULT_SPLIT_RULE = [
    {
        "stage": "main",
        "mode": "all",
        "description": "default main",
    },
]


def list_dataset_files(ds: dict[str, Any]) -> list[Path]:
    cache_dir = Path(ds["cache_dir"])
    if not cache_dir.exists():
        return []
    parquets = sorted(cache_dir.rglob("*.parquet"))
    jsonls = sorted(cache_dir.rglob("*.jsonl"))
    return [*parquets, *jsonls]


def split_rules_for_dataset(ds: dict[str, Any]) -> list[dict[str, Any]]:
    return SPLIT_RULES_BY_DATASET.get(ds["name"], DEFAULT_SPLIT_RULE)


def build_filter_mask(values: pa.Array | pa.ChunkedArray, rule: dict[str, Any]) -> pa.Array | pa.ChunkedArray:
    if rule["mode"] == "all":
        raise ValueError("mode=all 不需要过滤 mask")

    op = rule["op"]
    value = rule["value"]
    if op == "ge":
        mask = pc.greater_equal(values, value)
    elif op == "lt":
        mask = pc.less(values, value)
    elif op == "eq":
        mask = pc.equal(values, value)
    elif op == "ne":
        mask = pc.not_equal(values, value)
    else:
        raise ValueError(f"不支持的比较操作: {op}")

    if rule.get("include_null", False):
        mask = pc.or_(pc.is_null(values), mask)
    return pc.fill_null(mask, False)


def count_rows_in_jsonl(file_path: Path) -> int:
    with file_path.open("r", encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip())


def count_matching_rows(file_path: Path, rule: dict[str, Any]) -> int:
    if file_path.suffix == ".parquet":
        parquet_file = pq.ParquetFile(file_path)
        if rule["mode"] == "all":
            return parquet_file.metadata.num_rows

        total = 0
        for batch in parquet_file.iter_batches(columns=[rule["column"]], batch_size=65_536):
            values = batch.column(0)
            mask = build_filter_mask(values, rule)
            total += int((pc.sum(pc.cast(mask, pa.int64())).as_py()) or 0)
        return total

    if file_path.suffix == ".jsonl":
        if rule["mode"] != "all":
            raise ValueError(f"JSONL 暂不支持行级过滤: {file_path}")
        return count_rows_in_jsonl(file_path)

    return 0


def iter_filtered_texts(
    files: list[Path],
    *,
    text_field: str,
    rule: dict[str, Any],
    sample_limit: int,
) -> tuple[int, float]:
    sample_count = 0
    char_total = 0

    for file_path in files:
        if sample_count >= sample_limit:
            break

        if file_path.suffix == ".parquet":
            columns = [text_field]
            if rule["mode"] != "all":
                columns.append(rule["column"])

            parquet_file = pq.ParquetFile(file_path)
            for batch in parquet_file.iter_batches(columns=columns, batch_size=4_096):
                table = pa.Table.from_batches([batch])
                if text_field not in table.column_names:
                    raise KeyError(f"{file_path} 缺少文本列 {text_field}")
                if rule["mode"] != "all":
                    mask = build_filter_mask(table[rule["column"]], rule)
                    table = table.filter(mask)
                for value in table[text_field].to_pylist():
                    if value is None:
                        continue
                    text = str(value)
                    if not text:
                        continue
                    char_total += len(text)
                    sample_count += 1
                    if sample_count >= sample_limit:
                        break
                if sample_count >= sample_limit:
                    break
            continue

        if file_path.suffix == ".jsonl":
            if rule["mode"] != "all":
                raise ValueError(f"JSONL 暂不支持行级过滤: {file_path}")
            with file_path.open("r", encoding="utf-8") as handle:
                for line in handle:
                    if not line.strip():
                        continue
                    record = json.loads(line)
                    value = record.get(text_field)
                    if value is None:
                        continue
                    text = str(value)
                    if not text:
                        continue
                    char_total += len(text)
                    sample_count += 1
                    if sample_count >= sample_limit:
                        break

    avg_chars = char_total / max(1, sample_count)
    return sample_count, avg_chars


encode_tasks: list[dict[str, Any]] = []
estimate_rows: list[dict[str, Any]] = []

for ds in DATASETS:
    files = list_dataset_files(ds)
    if not files:
        print(f"[skip] {ds['name']}: 未找到 parquet/jsonl 文件")
        continue

    rules = split_rules_for_dataset(ds)
    for rule in rules:
        matched_rows = 0
        matched_files = 0
        for file_path in tqdm(files, desc=f"count {ds['name']} -> {rule['stage']}", unit="file", leave=False):
            rows = count_matching_rows(file_path, rule)
            if rows <= 0:
                continue
            matched_rows += rows
            matched_files += 1
            try:
                size = file_path.stat().st_size
            except OSError:
                size = 0
            encode_tasks.append({
                "dataset": ds["name"],
                "stage": rule["stage"],
                "category": ds["category"],
                "text_field": ds.get("text_field"),
                "path": str(file_path),
                "size": size,
                "matched_rows": rows,
                "row_filter": None if rule["mode"] == "all" else {
                    "column": rule["column"],
                    "op": rule["op"],
                    "value": rule["value"],
                    "include_null": rule.get("include_null", False),
                },
            })

        sampled_rows, avg_chars = iter_filtered_texts(
            files,
            text_field=ds.get("text_field") or "text",
            rule=rule,
            sample_limit=STAGE_ESTIMATE_SAMPLE_ROWS,
        )
        estimate_rows.append({
            "dataset": ds["name"],
            "category": ds["category"],
            "stage": rule["stage"],
            "rule": rule["description"],
            "files": matched_files,
            "rows": matched_rows,
            "sampled_rows": sampled_rows,
            "avg_chars/row": round(avg_chars, 2),
            "est_tokens": round((matched_rows * avg_chars) / STAGE_ESTIMATE_CHARS_PER_TOKEN),
        })

# 大文件优先，但过滤后的子集仍保留 matched_rows 供后续编码器使用
encode_tasks.sort(key=lambda task: (task["size"], task["matched_rows"]), reverse=True)

main_files = sum(1 for task in encode_tasks if task["stage"] == "main")
anneal_files = sum(1 for task in encode_tasks if task["stage"] == "anneal")
total_rows = sum(task["matched_rows"] for task in encode_tasks)

estimate_df = pd.DataFrame(estimate_rows).sort_values(["stage", "dataset", "rule"]).reset_index(drop=True)
total_est_tokens = int(estimate_df["est_tokens"].sum()) if not estimate_df.empty else 0
if not estimate_df.empty:
    estimate_df["est_tok_%"] = [
        f"{value / max(1, total_est_tokens) * 100:.1f}%"
        for value in estimate_df["est_tokens"]
    ]

stage_df = (
    estimate_df.groupby("stage")[["rows", "est_tokens"]]
    .sum()
    .reset_index()
    .sort_values("stage")
    if not estimate_df.empty else pd.DataFrame(columns=["stage", "rows", "est_tokens"])
)
if not stage_df.empty:
    stage_df["est_tok_%"] = [
        f"{value / max(1, total_est_tokens) * 100:.1f}%"
        for value in stage_df["est_tokens"]
    ]

print(f"encode tasks: total={len(encode_tasks)}  main={main_files}  anneal={anneal_files}")
print(f"matched rows : {total_rows:,}")
print(f"shards dir   : {SHARDS_DIR}")

print("\n── 分流明细（粗估） ───────────────────────────────────────────────")
display(estimate_df)

print("\n── 按阶段汇总（粗估 token） ───────────────────────────────────────")
display(stage_df)

count opc_annealing -> anneal:   0%|          | 0/191 [00:00<?, ?file/s]

count finemath_plus4 -> main:   0%|          | 0/64 [00:00<?, ?file/s]

count the_stack_v2_python -> anneal:   0%|          | 0/84 [00:00<?, ?file/s]

count the_stack_v2_python -> main:   0%|          | 0/84 [00:00<?, ?file/s]

count starcoderdata_python_edu -> anneal:   0%|          | 0/125 [00:00<?, ?file/s]

...

count fineweb_edu_100bt -> main:   0%|          | 0/57 [00:00<?, ?file/s]

encode tasks: total=550  main=330  anneal=220
matched rows : 14,152,473
shards dir   : /data/ldyData/LLM-Walk-Through/data/cache/walkie_code/intermediate/shards

── 分流明细（粗估） ───────────────────────────────────────────────


,dataset,category,stage,rule,files,rows,sampled_rows,avg_chars/row,est_tokens,est_tok_%
0,opc_annealing,code,anneal,full dataset,11,997044,20000,621.56,154931240,1.0%
1,starcoderdata_python_edu,code,anneal,int_score >= 4,125,548102,20000,2059.97,282268974,1.8%
2,the_stack_v2_python,code,anneal,star_events_count >= 30,84,599673,20000,7766.78,1164382343,7.5%
3,finemath_plus4,math,main,default main,64,978752,20000,5515.55,1349589571,8.7%
4,fineweb_edu_100bt,text,main,default main,57,2957575,20000,5222.74,3861663273,24.9%
5,starcoderdata_python_edu,code,main,int_score < 4 or null,125,3612060,20000,3988.69,3601848661,23.2%
6,the_stack_v2_python,code,main,star_events_count <30 or null,84,4459267,20000,4582.53,5108679139,32.9%



── 按阶段汇总（粗估 token） ───────────────────────────────────────


,stage,rows,est_tokens,est_tok_%
0,anneal,2144819,1601582557,10.3%
1,main,12007654,13921780644,89.7%


## 10. 编码成 bin

基于 Step 9 生成的 `encode_tasks` 直接导出 `main.bin` / `anneal.bin`，并额外切出 `main_val.bin` / `anneal_val.bin` 供训练阶段计算验证 loss。

In [64]:
# ── Step 10：导出编码配置并打印脚本命令 ───────────────────────────────
# 长时间的 bin 编码放到独立脚本里执行，避免 notebook 进程被大任务绑定。
# 脚本路径：scripts/encode_walkie.py
# 默认使用“每个 worker 初始化一次 tokenizer 并复用”的逻辑；
# 确认列干净后可加 --trust-text，省去热循环里的类型/空串判断。

FORCE_EXPORT_BINS    = True
ENCODE_BATCH_ROWS    = 4096
WORKER_RAYON_THREADS = 2
WORKER_ARROW_THREADS = 1
WRITE_CHUNK_MB       = 64
DTYPE                = np.dtype(np.uint16)
COPY_BUFFER_MB       = 1
TRUST_TEXT           = True   # 数据探索已确认 text_field 干净时可保持 True；否则改 False
MAIN_VAL_RATIO       = 0.002  # main 阶段验证集比例
ANNEAL_VAL_RATIO     = 0.01   # anneal 阶段验证集比例
VAL_SPLIT_SALT       = "walkie-val-v1"

if not encode_tasks:
    raise RuntimeError("encode_tasks 为空：请先运行 Step 9，并确认 cache_dir 下存在 parquet/jsonl 文件")

EOS_TOKEN = "<|endoftext|>"
PAD_TOKEN = "<|pad|>"
EOS_ID = hf_tok.token_to_id(EOS_TOKEN)
PAD_ID = hf_tok.token_to_id(PAD_TOKEN)
if EOS_ID is None or PAD_ID is None:
    raise RuntimeError(f"tokenizer 必须包含 {EOS_TOKEN} 与 {PAD_TOKEN}，请检查 Step 7")

_vocab_size = hf_tok.get_vocab_size()
dtype_limit = int(np.iinfo(DTYPE).max)
if _vocab_size - 1 > dtype_limit:
    raise RuntimeError(
        f"vocab_size={_vocab_size} 需要能表示 token id {_vocab_size - 1}，"
        f"但 dtype={DTYPE} 最大值只有 {dtype_limit}"
    )

SCRIPT_PATH       = ROOT / "scripts" / "encode_walkie.py"
TASKS_JSON        = OUTPUT_DIR / "encode_tasks_for_encode.json"
TOKENIZER_DIR     = TOKENIZER_PATH
MAIN_VAL_BIN      = OUTPUT_DIR / "main_val.bin"
ANNEAL_VAL_BIN    = OUTPUT_DIR / "anneal_val.bin"

tasks_payload = []
for task in encode_tasks:
    tasks_payload.append(
        {
            "dataset": task["dataset"],
            "stage": task["stage"],
            "text_field": task.get("text_field"),
            "path": task["path"],
            "size": int(task.get("size", 0)),
            "matched_rows": int(task.get("matched_rows", 0)),
            "row_filter": task.get("row_filter"),
        }
    )

TASKS_JSON.write_text(
    json.dumps(tasks_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

cmd_parts = [
    "uv run --extra walkie python scripts/encode_walkie.py",
    f'--tasks-json "{TASKS_JSON.as_posix()}"',
    f'--output-dir "{OUTPUT_DIR.as_posix()}"',
    f'--tokenizer-dir "{TOKENIZER_DIR.as_posix()}"',
    f'--main-bin "{MAIN_BIN.as_posix()}"',
    f'--anneal-bin "{ANNEAL_BIN.as_posix()}"',
    f'--main-val-bin "{MAIN_VAL_BIN.as_posix()}"',
    f'--anneal-val-bin "{ANNEAL_VAL_BIN.as_posix()}"',
    f'--meta "{META_PATH.as_posix()}"',
    f'--split-manifest "{SPLIT_MANIFEST_JSONL.as_posix()}"',
    f'--shards-dir "{SHARDS_DIR.as_posix()}"',
    f"--dtype {DTYPE}",
    f"--batch-rows {ENCODE_BATCH_ROWS}",
    f"--rayon-threads {WORKER_RAYON_THREADS}",
    f"--arrow-threads {WORKER_ARROW_THREADS}",
    f"--write-chunk-mb {WRITE_CHUNK_MB}",
    f"--copy-buffer-mb {COPY_BUFFER_MB}",
    f"--main-val-ratio {MAIN_VAL_RATIO}",
    f"--anneal-val-ratio {ANNEAL_VAL_RATIO}",
    f'--val-salt "{VAL_SPLIT_SALT}"',
]
if FORCE_EXPORT_BINS:
    cmd_parts.append("--force")
if TRUST_TEXT:
    cmd_parts.append("--trust-text")

ENCODE_CMD = " \\\n  ".join(cmd_parts)

print(f"tasks json    : {TASKS_JSON}")
print(f"script        : {SCRIPT_PATH}")
print(f"dtype         : {DTYPE}  vocab_size={_vocab_size}  eos_id={EOS_ID}  pad_id={PAD_ID}")
print(f"main val ratio: {MAIN_VAL_RATIO}")
print(f"anneal ratio  : {ANNEAL_VAL_RATIO}")
print("\n[run encode script]")
print(ENCODE_CMD)

print("\n[notes]")
print("- 脚本会直接消费 Step 9 生成的 encode_tasks，按 row_filter 真正分流 main / anneal。")
print("- 验证集按文档级稳定哈希切分，输出 main_val.bin / anneal_val.bin。")
print("- TRUST_TEXT=True 会跳过热循环里的类型/空串判断，速度更高。")
print("- 如果某个数据集列不干净，把 TRUST_TEXT 改为 False 后重新打印命令。")
print("- 脚本会写出四个 bin、split_manifest.jsonl 和 data_meta.json。")

tasks json    : /data/ldyData/LLM-Walk-Through/data/cache/walkie_code/encode_tasks_for_encode.json
script        : /data/ldyData/LLM-Walk-Through/scripts/encode_walkie.py
dtype         : uint16  vocab_size=65536  eos_id=0  pad_id=1
main val ratio: 0.002
anneal ratio  : 0.01

[run encode script]
uv run --extra walkie python scripts/encode_walkie.py \
  --tasks-json "/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/encode_tasks_for_encode.json" \
  --output-dir "/data/ldyData/LLM-Walk-Through/data/cache/walkie_code" \
  --tokenizer-dir "/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/tokenizer.json" \
  --main-bin "/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/main.bin" \
  --anneal-bin "/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/anneal.bin" \
  --main-val-bin "/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/main_val.bin" \
  --anneal-val-bin "/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/anneal_val.bin" \
  --meta "/data/ldyData/LLM-Walk-Throu

## 11. 验证与训练命令

用 `np.memmap` 打开 train / val 四个 bin 文件验证 token 数量，并打印可直接复制的训练命令。

In [65]:
main_mm = np.memmap(MAIN_BIN, dtype=DTYPE, mode="r")
main_val_mm = np.memmap(MAIN_VAL_BIN, dtype=DTYPE, mode="r")
anneal_mm = np.memmap(ANNEAL_BIN, dtype=DTYPE, mode="r")
anneal_val_mm = np.memmap(ANNEAL_VAL_BIN, dtype=DTYPE, mode="r")

print(f"main train tokens   = {len(main_mm):,}")
print(f"main val tokens     = {len(main_val_mm):,}")
print(f"anneal train tokens = {len(anneal_mm):,}")
print(f"anneal val tokens   = {len(anneal_val_mm):,}")

cmd = (
    "uv run torchrun --nproc-per-node=2 -m train.walkie_pretrain "
    "--config configs/train/pretrain_walkie.yaml "
    f"data.stages.main.bin={MAIN_BIN.as_posix()} "
    f"data.stages.main.val_bin={MAIN_VAL_BIN.as_posix()} "
    f"data.stages.main.dtype={DTYPE} "
    f"data.stages.anneal.bin={ANNEAL_BIN.as_posix()} "
    f"data.stages.anneal.val_bin={ANNEAL_VAL_BIN.as_posix()} "
    f"data.stages.anneal.dtype={DTYPE}"
)
print("\n[start training]")
print(cmd)

main train tokens   = 15,592,290,386
main val tokens     = 34,342,142
anneal train tokens = 1,926,617,773
anneal val tokens   = 23,185,178

[start training]
uv run torchrun --nproc-per-node=2 -m train.walkie_pretrain --config configs/train/pretrain_walkie.yaml data.stages.main.bin=/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/main.bin data.stages.main.val_bin=/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/main_val.bin data.stages.main.dtype=uint16 data.stages.anneal.bin=/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/anneal.bin data.stages.anneal.val_bin=/data/ldyData/LLM-Walk-Through/data/cache/walkie_code/anneal_val.bin data.stages.anneal.dtype=uint16
